# 05 — Rule-Based Business Insights

This notebook turns Phase 3b probabilities and Phase 4 SHAP contributions into transparent retail action labels for every held-out row. It loads the existing model and explainer logic and trains nothing. The actions are prioritization suggestions, not claims that a customer will purchase.

## Load model outputs and all-row SHAP contributions

The saved threshold-aware model provides each test probability and prediction at the cutoff loaded from its artifact. The Phase 4 explainer computes SHAP contributions for every held-out row without refitting the model. The executed output reports the actual cutoff, row count, feature count, and explainer type. Rule feature groups are discovered only from names in the fitted preprocessor.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.explain import load_explanation_context
from src.insights import (
    PROMOTION_MARGIN, STRONG_POSITIVE_SHAP, aggregate_action_summary,
    build_business_insights, compute_all_test_shap,
    render_business_insights_summary
)

results_dir = PROJECT_ROOT / 'results'
context = load_explanation_context(
    PROJECT_ROOT / 'models' / 'best_model.joblib',
    PROJECT_ROOT / 'data',
    results_dir / 'dataset_summary.txt'
)
shap_values, explainer_name = compute_all_test_shap(context)
print('Loaded classifier:', type(context['classifier']).__name__)
print('Tuned threshold:', f"{context['threshold']:.6f}")
print('Test rows:', len(context['data']['X_test']))
print('SHAP contribution shape:', shap_values.shape)
print('Explainer:', explainer_name)

Loaded classifier: LogisticRegression
Tuned threshold: 0.572596
Test rows: 5000
SHAP contribution shape: (5000, 52)
Explainer: LinearExplainer


## Inspectable action rules

Rules inspect only the three largest absolute SHAP drivers for a row. A positive product-category or location-frequency driver of at least +0.10 log-odds receives the specific recommendation action. Otherwise, probabilities at least 0.10 above the tuned threshold are higher-score promotion candidates; other threshold-qualified rows are ordinary promotion candidates. A below-threshold row with a positive engagement driver of at least +0.10 is marked for checkout/discount-friction investigation. Remaining below-threshold rows receive an engagement nudge. Rule order is fixed so every row receives exactly one action.

In [2]:
insights, groups = build_business_insights(context, shap_values)
print('Confirmed engagement features:', groups['engagement'])
print('Confirmed personalization features:', groups['personalization'])
print('Promotion margin:', PROMOTION_MARGIN)
print('Strong positive SHAP cutoff:', STRONG_POSITIVE_SHAP)
print('\nInsights table shape:', insights.shape)
print(insights.head(12).to_string(index=False, float_format=lambda value: f'{value:.6f}'))

Confirmed engagement features: ['pages_viewed', 'time_on_site_sec', 'added_to_cart', 'session_duration_bucket']
Confirmed personalization features: ['product_category_0', 'product_category_1', 'product_category_2', 'product_category_3', 'product_category_4', 'product_category_5', 'product_category_6', 'product_category_7', 'location_frequency']
Promotion margin: 0.1
Strong positive SHAP cutoff: 0.1

Insights table shape: (5000, 5)
 row_id  predicted_probability  tuned_threshold_prediction                                                                  top_drivers                     recommended_action
  16421               0.000627                           0   added_to_cart (-4.8886) | user_type_0 (+0.3862) | visit_season_0 (-0.1830)             Retention/engagement nudge
  16278               0.520797                           0 added_to_cart (+3.3972) | user_type_0 (-0.3160) | time_on_site_sec (-0.2020) Investigate checkout/discount friction
    813               0.000509          

## Aggregate actions and precision caveat

The tuned model's held-out precision is moderate. Therefore, `Candidate for promotion` means suitable for cautious, low-cost prioritization—not a guaranteed purchaser. A meaningful share of these flags will be false positives.

In [3]:
aggregate, caveat = aggregate_action_summary(insights, context)
print(aggregate.to_string(index=False, float_format=lambda value: f'{value:.2f}'))
print('\nPrecision caveat:')
print(caveat)
print('\nRows assigned exactly once:', len(insights) == aggregate['row_count'].sum())

                           recommended_action  row_count  percent_of_test_rows
      Personalized recommendation opportunity          0                  0.00
       Candidate for promotion - higher score       1341                 26.82
Candidate for promotion - threshold-qualified       1343                 26.86
       Investigate checkout/discount friction        576                 11.52
                   Retention/engagement nudge       1740                 34.80

Precision caveat:
At the tuned threshold, held-out precision is 0.3681. Therefore, candidate-for-promotion flags will include a meaningful share of false positives and should be treated as candidates for low-cost outreach, not as guaranteed purchasers.

Rows assigned exactly once: True


## Save the full insights table and summary

The CSV contains exactly the requested fields: source row ID, predicted probability, tuned-threshold prediction, top three signed SHAP drivers, and recommended action. The text summary records the actual rule groups, rule order, action counts, and precision warning.

In [4]:
table_path = results_dir / 'tables' / 'business_insights.csv'
insights.to_csv(table_path, index=False, float_format='%.8f')
summary = render_business_insights_summary(
    context, explainer_name, groups, aggregate, caveat, table_path
)
summary_path = results_dir / 'business_insights_summary.txt'
summary_path.write_text(summary, encoding='utf-8')
print('Saved insights table:', table_path.resolve())
print('Saved Phase 5 summary:', summary_path.resolve())
print('No adaptive experiment was performed.')

Saved insights table: P:\College\Sem VII Acad\IPRM\Project\results\tables\business_insights.csv
Saved Phase 5 summary: P:\College\Sem VII Acad\IPRM\Project\results\business_insights_summary.txt
No adaptive experiment was performed.


## Phase 5 boundary

Rule-based action mapping is complete. No adaptive experiment, reinforcement learning, or additional predictive model is included.